<a href="https://colab.research.google.com/github/MoneyTreeNJKim/NJKim/blob/main/emg-robot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Colab 셀 0: 파이썬 버전 확인과 환경 선택
!python --version
!pip -q install uv
# 3.12.x 이면 아래 두 줄은 건너뜀. 아니면 실행:
!uv python install 3.12 && uv venv /content/venv --python 3.12
# 이후 모든 !python 을 !/content/venv/bin/python 으로, !pip 를 !uv pip install --python /content/venv/bin/python 으로 바꿈

Python 3.13.16
Installed Python 3.12.16 in 1.19s
 + cpython-3.12.16-linux-x86_64-gnu (python3.12)
Using CPython 3.12.16
Creating virtual environment at: venv
Activate with: source venv/bin/activate


In [2]:
# Colab 셀 1 (T4 런타임)
from google.colab import drive; drive.mount('/content/drive')
%pip install -q py-soma-x

import torch
# py-soma-x의 실제 모듈 이름은 soma입니다.
import soma
print('soma', soma.__version__ if hasattr(soma,'__version__') else 'ok', torch.cuda.is_available())

Mounted at /content/drive
soma 0.3.6 True


In [3]:
# Colab 셀 2: SOMAHandLayer 가 돌아가는지 확인 (디바이스 설정 적용)
import torch
from soma import SOMAHandLayer

# GPU 장치 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 모델 생성 및 장치 이동
hand = SOMAHandLayer(hand_type='right', lod='mid', device=device)

# 텐서 생성 시 디바이스를 명시적으로 지정
pose = torch.zeros(1, 25, 3, device=device)
ident = torch.zeros(1, hand.num_shape_components, device=device)

# 연산 수행
out = hand(pose, ident)
print("Vertices shape:", out['vertices'].shape, "\nJoints shape:", out['joints'].shape)

Using device: cuda
Warp 1.19.0 initialized:
   CUDA Toolkit 13.4, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool supported)
   Kernel cache:
     /root/.cache/warp/1.19.0


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/serialization.py:2244: UserWarning: Validating sparse tensor invariants because weights_only=True; this is an O(nnz) scan per sparse tensor and may be slow for large checkpoints.
  torch._utils._validate_loaded_sparse_tensors(weights_only=weights_only)


Module soma.geometry.align_vectors_warp 924fca9 load on device 'cuda:0' took 12995.96 ms  (compiled)
Module soma.geometry.lbs_warp 1cfb16a load on device 'cuda:0' took 1013.33 ms  (compiled)
Vertices shape: torch.Size([1, 2859, 3]) 
Joints shape: torch.Size([1, 25, 3])


In [4]:
# Colab 셀 3: somehand
!git clone --recursive https://github.com/BotRunner64/somehand.git /content/somehand
%cd /content/somehand
!pip -q install -e .
!ls configs/retargeting | head -n 30   # 지원 로봇 손 YAML 목록
!python -c "import somehand; print('somehand ok')"

Cloning into '/content/somehand'...
remote: Enumerating objects: 1983, done.
remote: Counting objects: 100% (402/402), done.
remote: Compressing objects: 100% (254/254), done.
remote: Total 1983 (delta 185), reused 288 (delta 126), pack-reused 1581 (from 1)
Receiving objects: 100% (1983/1983), 35.44 MiB | 16.22 MiB/s, done.
Resolving deltas: 100% (1161/1161), done.
Submodule 'third_party/linkerhand-python-sdk' (https://github.com/BotRunner64/linkerhand-python-sdk.git) registered for path 'third_party/linkerhand-python-sdk'
Cloning into '/content/somehand/third_party/linkerhand-python-sdk'...
remote: Enumerating objects: 2437, done.        
remote: Counting objects: 100% (375/375), done.        
remote: Compressing objects: 100% (111/111), done.        
remote: Total 2437 (delta 307), reused 301 (delta 264), pack-reused 2062 (from 1)        
Receiving objects: 100% (2437/2437), 86.77 MiB | 16.85 MiB/s, done.
Resolving deltas: 100% (1458/1458), done.
Submodule path 'third_party/linkerhan

In [5]:
import yaml
import os
from pathlib import Path

# configs/retargeting 디렉토리에서 right 폴더 내의 yaml 파일들을 탐색합니다.
base_dir = Path("/content/somehand/configs/retargeting")
right_yaml_files = list(base_dir.glob("right/**/*.yaml"))

if right_yaml_files:
    # 첫 번째 우측 손 설정 파일 로드 예제
    config_path = right_yaml_files[0]
    with open(config_path, "r") as f:
        hand_config = yaml.safe_load(f)

    print("=== Robot Hand Configuration Loaded ===")
    print(f"File path: {config_path.relative_to(base_dir.parent.parent)}")
    print("\n[Configuration Keys and Values (Partial)]")
    for key in list(hand_config.keys())[:10]:
        print(f"- {key}: {hand_config[key]}")
else:
    print(f"'right' 폴더 내부에서 YAML 설정 파일을 찾을 수 없습니다.")

=== Robot Hand Configuration Loaded ===
File path: configs/retargeting/right/inspire_ftp_right.yaml

[Configuration Keys and Values (Partial)]
- extends: ../base/inspire_ftp.yaml
- hand: {'name': 'inspire_ftp_right', 'side': 'right', 'mjcf_path': '../../../assets/mjcf/inspire_ftp_right/model.xml'}


In [6]:
base_config_path = (config_path.parent / hand_config['extends']).resolve()

if base_config_path.exists():
    print(f"=== Base Configuration Content ({base_config_path.name}) ===\n")
    with open(base_config_path, "r") as f:
        print(f.read())
else:
    print(f"상속할 베이스 설정 파일을 찾을 수 없습니다: {base_config_path}")

=== Base Configuration Content (inspire_ftp.yaml) ===

extends: "./_universal_common.yaml"

retargeting:
  vector_constraints:
    - human: [1, 2]
      robot: ["thumb_2", "thumb_3"]
    - human: [2, 3]
      robot: ["thumb_3", "thumb_4"]
    - human: [3, 4]
      robot: ["thumb_4", "thumb_4_tip"]
      robot_types: ["body", "site"]
    - human: [5, 7]
      robot: ["index_1", "index_2"]
    - human: [7, 8]
      robot: ["index_2", "index_2_tip"]
      robot_types: ["body", "site"]
    - human: [9, 11]
      robot: ["middle_1", "middle_2"]
    - human: [11, 12]
      robot: ["middle_2", "middle_2_tip"]
      robot_types: ["body", "site"]
    - human: [13, 15]
      robot: ["ring_1", "ring_2"]
    - human: [15, 16]
      robot: ["ring_2", "ring_2_tip"]
      robot_types: ["body", "site"]
    - human: [17, 19]
      robot: ["little_1", "little_2"]
    - human: [19, 20]
      robot: ["little_2", "little_2_tip"]
      robot_types: ["body", "site"]
  distance_constraints:
    - human: [4, 8

## **Colab: SOMA-X와 somehand**

In [ ]:
# Colab 셀 0: 파이썬 버전 확인과 환경 선택
!python --version
!pip -q install uv
# 3.12.x 이면 아래 두 줄은 건너뜀. 아니면 실행:
!uv python install 3.12 && uv venv /content/venv --python 3.12
# 이후 모든 !python 을 !/content/venv/bin/python 으로, !pip 를 !uv pip install --python /content/venv/bin/python 으로 바꿈

Python 3.13.16
Python 3.12 is already installed
Using CPython 3.12.16
Creating virtual environment at: /content/venv
? A virtual environment already exists at `/content/venv`. Do you want to replace it? [y/n] › yes

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to skip this prompt

In [ ]:
# Colab 셀 1 (T4 런타임)
from google.colab import drive; drive.mount('/content/drive')
!pip -q install py-soma-x
import soma, torch   # 모듈 이름은 soma
print('soma-x', soma.__version__, torch.cuda.is_available())

In [ ]:
# Colab 셀 2: SOMAHandLayer 가 돌아가는지 확인(자산은 첫 호출 때 HF 에서 자동 내려받음)
from soma import SOMAHandLayer
hand = SOMAHandLayer(hand_type='right', device='cuda', lod='mid')   # 기본 mode='warp'(GPU)
pose = torch.zeros(1, 25, 3, device='cuda'); ident = torch.zeros(1, hand.num_shape_components, device='cuda')
out = hand(pose, ident)
print(out.vertices.shape, out.joints.shape)   # (1, V, 3), (1, 25, 3)

In [ ]:
# Colab 셀 3: somehand
!git clone --recursive https://github.com/BotRunner64/somehand.git /content/somehand
%cd /content/somehand && pip -q install -e .
!ls configs/retargeting | head -n 30   # 지원 로봇 손 YAML 목록
!python -c "import somehand; print('somehand ok')"